In [3]:
import pandas as pd
import numpy as np

In [4]:
ratings = pd.read_csv("../data/ratings.csv")
movies = pd.read_csv("../data/movies.csv")

In [5]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [6]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
print("Ratings Shape:", ratings.shape)
print("Movies Shape:", movies.shape)

Ratings Shape: (100836, 4)
Movies Shape: (9742, 3)


### ratings.csv

- userId → User identifier

- movieId → Movie identifier

- rating → User rating (1–5)

- timestamp → When rated

### movies.csv

- movieId → Movie identifier

- title → Movie name

- genres → Categories (Action, Comedy…)

In [8]:
#timestamp is not needed for recommendations.
ratings = ratings.drop(columns=["timestamp"])
ratings.head()

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


In [9]:
ratings.isnull().sum()

userId     0
movieId    0
rating     0
dtype: int64

In [10]:
movies.isna().sum()

movieId    0
title      0
genres     0
dtype: int64

In [11]:
#Clean Movie Titles

movies['clean_title'] = movies['title'].str.replace(r"\(\d{4}\)", "", regex=True)
movies['clean_title'] = movies['clean_title'].str.strip()

movies[['title','clean_title']].head()

,title,clean_title
0,Toy Story (1995),Toy Story
1,Jumanji (1995),Jumanji
2,Grumpier Old Men (1995),Grumpier Old Men
3,Waiting to Exhale (1995),Waiting to Exhale
4,Father of the Bride Part II (1995),Father of the Bride Part II


In [12]:
#Prepare Text for Embeddings
#We combine useful text fields.

movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)

movies['combined_text'] = movies['clean_title'] + " " + movies['genres']

movies[['clean_title','genres','combined_text']].head()

,clean_title,genres,combined_text
0,Toy Story,Adventure Animation Children Comedy Fantasy,Toy Story Adventure Animation Children Comedy ...
1,Jumanji,Adventure Children Fantasy,Jumanji Adventure Children Fantasy
2,Grumpier Old Men,Comedy Romance,Grumpier Old Men Comedy Romance
3,Waiting to Exhale,Comedy Drama Romance,Waiting to Exhale Comedy Drama Romance
4,Father of the Bride Part II,Comedy,Father of the Bride Part II Comedy


In [13]:
#Merge Ratings + Movies

data = ratings.merge(movies, on="movieId")
data.head()


,userId,movieId,rating,title,genres,clean_title,combined_text
0,1,1,4.0,Toy Story (1995),Adventure Animation Children Comedy Fantasy,Toy Story,Toy Story Adventure Animation Children Comedy ...
1,1,3,4.0,Grumpier Old Men (1995),Comedy Romance,Grumpier Old Men,Grumpier Old Men Comedy Romance
2,1,6,4.0,Heat (1995),Action Crime Thriller,Heat,Heat Action Crime Thriller
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),Mystery Thriller,Seven (a.k.a. Se7en),Seven (a.k.a. Se7en) Mystery Thriller
4,1,50,5.0,"Usual Suspects, The (1995)",Crime Mystery Thriller,"Usual Suspects, The","Usual Suspects, The Crime Mystery Thriller"


In [14]:
data.shape

(100836, 7)

In [15]:
#Basic Sanity Checks

print("Unique Users:", ratings['userId'].nunique())
print("Unique Movies:", ratings['movieId'].nunique())
print("Total Ratings:", len(ratings))

Unique Users: 610
Unique Movies: 9724
Total Ratings: 100836


In [16]:
#Load Transformer Model
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

d:\hybrid-product-recommender\myvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3378.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
# Generate Embeddings

texts = movies['combined_text'].tolist()

item_embeddings = model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 305/305 [00:24<00:00, 12.48it/s]


In [18]:
item_embeddings.shape

(9742, 384)

In [19]:
# Quick Similarity Test

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(item_embeddings)

In [20]:
similarity_matrix

array([[1.0000001 , 0.60759807, 0.35230196, ..., 0.2727608 , 0.38529864,
        0.31889033],
       [0.60759807, 1.        , 0.16245428, ..., 0.1987539 , 0.23908225,
        0.18749955],
       [0.35230196, 0.16245428, 1.0000002 , ..., 0.27890605, 0.20509383,
        0.40340546],
       ...,
       [0.2727608 , 0.1987539 , 0.27890605, ..., 1.0000001 , 0.16386622,
        0.26046306],
       [0.38529864, 0.23908225, 0.20509383, ..., 0.16386622, 1.0000002 ,
        0.19385085],
       [0.31889033, 0.18749955, 0.40340546, ..., 0.26046306, 0.19385085,
        1.0000002 ]], dtype=float32)

### Cosine Similarity

$$
\text{Cosine Similarity} =
\frac{\sum_{i=1}^{n} A_i B_i}
{\sqrt{\sum_{i=1}^{n} A_i^2} \; \sqrt{\sum_{i=1}^{n} B_i^2}}
$$

In [21]:
movie_index = 0
movies.iloc[movie_index]['clean_title']

'Toy Story'

In [22]:
similar_scores = list(enumerate(similarity_matrix[movie_index]))
similar_scores = sorted(similar_scores, key=lambda x: x[1], reverse=True)

top_indices = [i[0] for i in similar_scores[1:6]]

movies.iloc[top_indices][['clean_title','genres']]

,clean_title,genres
2355,Toy Story 2,Adventure Animation Children Comedy Fantasy
8357,The Lego Movie,Action Adventure Animation Children Comedy Fan...
7355,Toy Story 3,Adventure Animation Children Comedy Fantasy IMAX
8219,Turbo,Adventure Animation Children Comedy Fantasy
6194,"Wild, The",Adventure Animation Children Comedy Fantasy


In [23]:
import pickle

# Save embeddings
with open("../models/item_embeddings.pkl", "wb") as f:
    pickle.dump(item_embeddings, f)

# Save movie ids
with open("../models/movie_meta.pkl", "wb") as f:
    pickle.dump(movies[['movieId','clean_title','genres']], f)

In [24]:
import pickle
import numpy as np

# Load item embeddings
with open("../models/item_embeddings.pkl", "rb") as f:
    item_embeddings = pickle.load(f)

# Load movie metadata
with open("../models/movie_meta.pkl", "rb") as f:
    movie_meta = pickle.load(f)

In [25]:
# Build MovieId → Index Mapping
movie_id_to_index = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_meta['movieId'])
}

### User Vector (Weighted Average of Movie Embeddings)

$$
\text{UserVec} =
\frac{
\sum (\text{rating} \times \text{movie\_embedding})
}{
\sum \text{ratings}
}
$$

In [26]:
ratings.head(1)

,userId,movieId,rating
0,1,1,4.0


In [27]:
# Function to Create One User Embedding

def create_user_embedding(user_id, ratings_df):
    
    user_ratings = ratings_df[ratings_df['userId'] == user_id]
    
    if user_ratings.empty:
        return None
    
    vectors = []
    weights = []
    
    for _, row in user_ratings.iterrows():
        movie_id = row['movieId']
        rating = row['rating']
        
        if movie_id in movie_id_to_index:
            idx = movie_id_to_index[movie_id]
            vectors.append(item_embeddings[idx])
            weights.append(rating)
    
    if not vectors:
        return None
    
    vectors = np.array(vectors)
    weights = np.array(weights)
    
    user_vector = np.average(vectors, axis=0, weights=weights)
    
    return user_vector


In [28]:
sample_user = ratings['userId'].iloc[0]

user_vector = create_user_embedding(sample_user, ratings)

user_vector.shape

(384,)

In [29]:
# Build Embeddings for All Users
user_embeddings = {}

for user_id in ratings['userId'].unique():
    vec = create_user_embedding(user_id, ratings)
    if vec is not None:
        user_embeddings[user_id] = vec

len(user_embeddings)

610

In [30]:
with open("../models/user_embeddings.pkl", "wb") as f:
    pickle.dump(user_embeddings, f)

In [31]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

with open("../models/user_embeddings.pkl", "rb") as f:
    user_embeddings = pickle.load(f)

In [32]:
# Recommendation Function

def recommend_movies(user_id, top_n=10):
    
    if user_id not in user_embeddings:
        print("Cold start user")
        return None
    
    user_vector = user_embeddings[user_id]
    
    # Compute similarity with all movies
    scores = cosine_similarity(
        [user_vector],
        item_embeddings
    )[0]
    
    # Get top movie indices
    top_indices = np.argsort(scores)[-top_n:][::-1]
    
    # Fetch movie details
    recommendations = movie_meta.iloc[top_indices].copy()
    recommendations['similarity_score'] = scores[top_indices]
    
    return recommendations

In [33]:
# Test Recommendations

sample_user = list(user_embeddings.keys())[0]

recommend_movies(sample_user, top_n=10)

,movieId,clean_title,genres,similarity_score
3069,4121,Innerspace,Action Adventure Comedy Sci-Fi,0.793957
6580,55247,Into the Wild,Action Adventure Drama,0.792773
7617,87194,The Way,Adventure Comedy Drama,0.783461
6398,50798,Epic Movie,Adventure Comedy,0.779306
7002,67734,Adventureland,Comedy Drama,0.776451
1341,1824,Homegrown,Comedy Thriller,0.775293
6274,47538,Crime Busters,Action Adventure Comedy Crime,0.773854
8575,116897,Wild Tales,Comedy Drama Thriller,0.770597
3263,4410,Something Wild,Comedy Crime Drama,0.769789
422,485,Last Action Hero,Action Adventure Comedy Fantasy,0.769464


In [34]:
# Train/Test Split
from sklearn.model_selection import train_test_split

train_ratings, test_ratings = train_test_split(
    ratings,
    test_size=0.3,
    random_state=42
)

In [35]:
# Rebuild User Embeddings Using Train Data Only
train_user_embeddings = {}

for user_id in train_ratings['userId'].unique():
    vec = create_user_embedding(user_id, train_ratings)
    if vec is not None:
        train_user_embeddings[user_id] = vec

### Precision@K

$$
\text{Precision@K} =
\frac{
\text{Relevant Recommendations}
}{
K
}
$$

In [36]:
# Precision@K Function

def precision_at_k(user_id, k=10, threshold=3.0):
    
    if user_id not in train_user_embeddings:
        return None
    
    user_vector = train_user_embeddings[user_id]
    
    scores = cosine_similarity([user_vector], item_embeddings)[0]
    top_indices = np.argsort(scores)[-k:][::-1]
    
    seen_movies = train_ratings[train_ratings['userId'] == user_id]['movieId'].values

    recommended = movie_meta.iloc[top_indices]
    recommended = recommended[~recommended['movieId'].isin(seen_movies)]

    recommended_movie_ids = recommended['movieId'].values[:k]
    
    # Movies user liked in test set
    user_test = test_ratings[test_ratings['userId'] == user_id]
    relevant_movies = user_test[user_test['rating'] >= threshold]['movieId'].values
    
    if len(relevant_movies) == 0:
        return None
    
    hits = len(set(recommended_movie_ids) & set(relevant_movies))
    
    return hits / k

In [37]:
sample_user = list(train_user_embeddings.keys())[0]
precision_at_k(sample_user, k=10)

0.1

In [38]:
# Average Precision@K Across Users
precisions = []

for user_id in train_user_embeddings.keys():
    p = precision_at_k(user_id, k=10)
    if p is not None:
        precisions.append(p)

np.mean(precisions)

0.011184210526315791

In [39]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
import numpy as np

In [ ]:
# Prepare Data for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# 🎯 What is SVD in Recommender Systems?

**SVD (Singular Value Decomposition)** is an algorithm used to  
predict how much a user will like a movie or product.

It is one of the most popular techniques in recommendation systems.

---

## 🧠 Simple Idea

Instead of using a huge table of ratings directly,  
SVD learns:

- Hidden features of users
- Hidden features of items (movies/products)

These hidden features are called **latent factors**.

---

## 🎬 Example

User–Movie Ratings:

| User | Movie A | Movie B | Movie C |
|------|---------|---------|---------|
| U1   | 5       | ?       | 3       |
| U2   | 4       | 2       | ?       |
| U3   | ?       | 5       | 4       |

`?` = Missing ratings

SVD helps to:
- Understand user preferences
- Understand movie characteristics
- Predict missing ratings

---

## ⚙️ What SVD Does

It breaks the big rating matrix into smaller matrices:

R ≈ U × Σ × Vᵀ

Where:

- **R** → Original rating matrix  
- **U** → User feature matrix  
- **Σ** → Importance weights  
- **Vᵀ** → Item feature matrix  

---

## 🎯 Rating Prediction

Predicted Rating =  
(User Features) · (Item Features)

Dot product of user and item vectors.

---

## ✅ Why SVD is Useful

- Handles missing ratings
- Learns user taste automatically
- Reduces data size
- Gives personalized recommendations

---

## 💻 Using SVD in Python (Surprise Library)

```python
from surprise import SVD

model = SVD()
model.fit(trainset)

prediction = model.predict(user_id, movie_id)
print(prediction.est)

In [41]:
cf_model = SVD()
cf_model.fit(trainset)

In [ ]:
# Predict Ratings for All Movies
def get_cf_scores(user_id):
    scores = []
    
    for movie_id in movie_meta['movieId']:
        pred = cf_model.predict(user_id, movie_id)
        scores.append(pred.est)
    
    return np.array(scores)

In [45]:
# Normalize Scores

'''
CF ratings ≈ 1–5
Embedding similarity ≈ -1 to 1

We scale both to 0–1.
'''
def minmax(x):
    return (x - x.min()) / (x.max() - x.min())

## 🎯 Hybrid Recommendation Score

### 📌 Final Score Formula

Final Score = α × CF Score + (1 − α) × Embedding Score

---

### 🧠 Where:

• **CF Score** → Collaborative Filtering score (from SVD)  
• **Embedding Score** → Content-based similarity score  
• **α (alpha)** → Weight between 0 and 1  

---

### 🎛️ Understanding Alpha

• α = 1   → Only Collaborative Filtering  
• α = 0   → Only Embedding-Based  
• α = 0.5 → Equal importance to both  

---

### 💻 Code

```python
final_scores = alpha * cf_scores + (1 - alpha) * emb_scores

In [46]:
def hybrid_recommend(user_id, top_n=10, alpha=0.7):
    
    if user_id not in user_embeddings:
        print("Cold start user")
        return None
    
    # Embedding score
    user_vector = user_embeddings[user_id]
    emb_scores = cosine_similarity([user_vector], item_embeddings)[0]
    
    # CF score
    cf_scores = get_cf_scores(user_id)
    
    # Normalize
    emb_scores = minmax(emb_scores)
    cf_scores = minmax(cf_scores)
    
    # Hybrid score
    final_scores = alpha * cf_scores + (1 - alpha) * emb_scores
    
    # Rank
    top_indices = np.argsort(final_scores)[-top_n:][::-1]
    
    recs = movie_meta.iloc[top_indices].copy()
    recs['hybrid_score'] = final_scores[top_indices]
    
    return recs

In [47]:
sample_user = list(user_embeddings.keys())[0]
hybrid_recommend(sample_user, top_n=10)

,movieId,clean_title,genres,hybrid_score
6710,58559,"Dark Knight, The",Action Crime Drama IMAX,0.963491
6676,57669,In Bruges,Comedy Crime Drama Thriller,0.948734
969,1270,Back to the Future,Adventure Comedy Sci-Fi,0.941471
4360,6377,Finding Nemo,Adventure Animation Children Comedy,0.939063
314,356,Forrest Gump,Comedy Drama Romance War,0.935196
277,318,"Shawshank Redemption, The",Crime Drama,0.931489
2226,2959,Fight Club,Action Crime Drama Thriller,0.927640
900,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action Adventure,0.923649
6315,48516,"Departed, The",Crime Drama Thriller,0.922331
46,50,"Usual Suspects, The",Crime Mystery Thriller,0.921187


In [48]:
def hybrid_precision_at_k(user_id, k=10, threshold=3.5):
    
    # Skip cold-start users
    if user_id not in train_user_embeddings:
        return None
    
    # Get hybrid recommendations
    recs = hybrid_recommend(user_id, top_n=50)  # get more, filter later
    if recs is None:
        return None
    
    # Remove already seen movies
    seen = train_ratings[train_ratings['userId'] == user_id]['movieId'].values
    recs = recs[~recs['movieId'].isin(seen)]
    
    recommended_movie_ids = recs['movieId'].values[:k]
    
    # Relevant movies in test set
    user_test = test_ratings[test_ratings['userId'] == user_id]
    relevant = user_test[user_test['rating'] >= threshold]['movieId'].values
    
    if len(relevant) == 0:
        return None
    
    hits = len(set(recommended_movie_ids) & set(relevant))
    
    return hits / k

In [49]:
sample_user = list(train_user_embeddings.keys())[0]
hybrid_precision_at_k(sample_user, k=10)

0.5

In [50]:
hybrid_precisions = []

for user_id in train_user_embeddings.keys():
    p = hybrid_precision_at_k(user_id, k=10)
    if p is not None:
        hybrid_precisions.append(p)

np.mean(hybrid_precisions)

0.17149917627677103

Improved recommendation precision by 15× using a hybrid ranking model combining transformer-based semantic embeddings with matrix factorization.

Embedding models capture semantic similarity, while collaborative filtering captures behavioral patterns. Combining both significantly improved ranking precision

# ❄️ Cold Start Strategy in Recommender Systems

## 🧠 What is Cold Start?

**Cold Start** happens when the recommender system does not have enough data  
to make accurate recommendations.

---

## ❄️ Case 1 — New User (User Cold Start)

The user has:

- No rating history  
- No interaction data  
- No past behavior  

👉 The system does not know the user’s preferences.

---

## ❄️ Case 2 — New Movie (Item Cold Start)

The movie has:

- No ratings  
- No user interactions  
- No popularity data  

👉 Collaborative filtering cannot score or recommend it.

---

## 🚩 Why Cold Start is a Problem

Collaborative Filtering depends on:

- Past user ratings  
- User–item interactions  

Without data → No similarity → No recommendations

---

## ✅ Common Solutions

### 👤 For New Users
- Ask users to rate a few items at signup
- Recommend trending/popular items
- Use demographic data

### 🎬 For New Movies
- Use content-based filtering
- Use movie metadata (genre, cast, description)
- Use embedding similarity

---

## 🚀 In Hybrid Recommender Systems

Cold Start is handled by:

- **Collaborative Filtering** → When history exists  
- **Content-Based / Embeddings** → When history is missing  

Together → System works even with new users or items

## ❄️ Case 1 — New User Strategy

### 🎯 Goal
Recommend good movies even if the user is new  
and has no rating history.

---

## ✅ Strategy 1 — Popularity-Based Recommendations

Recommend movies that:

- ⭐ Have many ratings (high popularity)
- ⭐ Have high average rating (good quality)

👉 These movies are generally liked by most users.

---

### 💡 Why This Works

- No user data needed
- Simple and reliable
- Good first-time user experience
- Widely used in real systems

---

### 🧠 Example Logic

Popularity Score can be based on:

Popularity Score = (Average Rating) × (Number of Ratings)

Higher score → Higher rank in recommendations

---

### 📌 Summary

When user history is missing:

➡ Recommend trending and highly-rated movies  
➡ Safe and effective starting point

In [ ]:


# Compute Popular Movies
popular_movies = (
    ratings.groupby('movieId')
    .agg(avg_rating=('rating','mean'),
         rating_count=('rating','count'))
    .reset_index()
)

# Keep movies with enough ratings
popular_movies = popular_movies[popular_movies['rating_count'] > 50]

# Score = rating × popularity
popular_movies['popularity_score'] = (
    popular_movies['avg_rating'] * np.log1p(popular_movies['rating_count'])
)

popular_movies = popular_movies.sort_values(
    by='popularity_score',
    ascending=False
)

In [53]:
# Function for New Users
def recommend_new_user(top_n=10):
    
    top_movie_ids = popular_movies['movieId'].values[:top_n]
    recs = movie_meta[movie_meta['movieId'].isin(top_movie_ids)].copy()
    
    return recs

In [54]:
recommend_new_user()

,movieId,clean_title,genres
224,260,Star Wars: Episode IV - A New Hope,Action Adventure Sci-Fi
257,296,Pulp Fiction,Comedy Crime Drama Thriller
277,318,"Shawshank Redemption, The",Crime Drama
314,356,Forrest Gump,Comedy Drama Romance War
461,527,Schindler's List,Drama War
510,593,"Silence of the Lambs, The",Crime Horror Thriller
659,858,"Godfather, The",Crime Drama
898,1196,Star Wars: Episode V - The Empire Strikes Back,Action Adventure Sci-Fi
1939,2571,"Matrix, The",Action Sci-Fi Thriller
2226,2959,Fight Club,Action Crime Drama Thriller


## ❄️ Case 2 — New Movie Strategy

### 🎬 Problem
A new movie is added to the system but it has:

- ❌ No ratings  
- ❌ No user interactions  
- ❌ No Collaborative Filtering (CF) score  

So CF methods cannot recommend it.

---

### ✅ Available Information

Even without ratings, we still have:

- 🧠 Text data (title, description, genre, tags)
- 🧠 Embeddings generated from this content

---

### 🚀 Solution — Embedding Similarity Only

We recommend the new movie using:

**Content-Based Filtering**

Compare the movie’s embedding with embeddings of other movies.

If embeddings are similar → Movies are similar.

---

### 🧠 How It Works

1. Convert movie text into embedding vectors  
2. Measure similarity (e.g., cosine similarity)  
3. Recommend to users who liked similar movies  

---

### 📌 Summary

When ratings are missing:

➡ Skip Collaborative Filtering  
➡ Use Embedding Similarity  
➡ Enables recommendation of new items

In [55]:
def similar_movies(movie_id, top_n=10):
    
    if movie_id not in movie_id_to_index:
        print("Movie not found")
        return None
    
    idx = movie_id_to_index[movie_id]
    
    sims = cosine_similarity(
        [item_embeddings[idx]],
        item_embeddings
    )[0]
    
    top_indices = np.argsort(sims)[-top_n-1:][::-1]
    
    recs = movie_meta.iloc[top_indices]
    recs = recs[recs['movieId'] != movie_id]
    
    return recs.head(top_n)

In [56]:
sample_movie = movie_meta['movieId'].iloc[0]
similar_movies(sample_movie)

,movieId,clean_title,genres
2355,3114,Toy Story 2,Adventure Animation Children Comedy Fantasy
8357,108932,The Lego Movie,Action Adventure Animation Children Comedy Fan...
7355,78499,Toy Story 3,Adventure Animation Children Comedy Fantasy IMAX
8219,103755,Turbo,Adventure Animation Children Comedy Fantasy
6194,45074,"Wild, The",Adventure Animation Children Comedy Fantasy
1706,2294,Antz,Adventure Animation Children Comedy Fantasy
1674,2253,Toys,Comedy Fantasy
8806,130520,Home,Adventure Animation Children Comedy Fantasy Sc...
8900,134853,Inside Out,Adventure Animation Children Comedy Drama Fantasy
3745,5218,Ice Age,Adventure Animation Children Comedy


#### Handled cold-start scenarios using popularity-based ranking for new users and embedding similarity for new items.